# Exporting Waveforms as Sigrok SR Files

**Scott Prahl**

**Apr 2026**

In [ ]:
%config InlineBackend.figure_format = 'retina'

import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

repo = "https://raw.githubusercontent.com/scottprahl/RigolWFM/main/tests/files/"


def sample_url(relative_path: str) -> str:
    return repo + relative_path


def download_sample(relative_path: str, directory: str | Path) -> Path:
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    out = directory / Path(relative_path).name
    urllib.request.urlretrieve(sample_url(relative_path), out)
    return out


def create_sigrok_session(relative_path: str, directory: str | Path) -> Path:
    directory = Path(directory)
    infile = download_sample(relative_path, directory)
    cmd = [
        sys.executable,
        "-m",
        "RigolWFM.wfmconvert",
        "--force",
        "--output-dir",
        str(directory),
        "sigrok",
        str(infile),
    ]
    subprocess.run(cmd, check=True, capture_output=True, text=True)
    session = directory / f"{infile.stem}.sr"
    if not session.exists():
        raise FileNotFoundError(f"Sigrok output was not created: {session}")
    return session


def read_member_text(session_path: str | Path, member: str) -> str:
    with zipfile.ZipFile(session_path) as zf:
        return zf.read(member).decode("utf-8", errors="replace")


SIGROK_AVAILABLE = shutil.which("sigrok-cli") is not None
SIGROK_AVAILABLE

## Introduction

A sigrok session file (`.sr`) is the native archive format used by PulseView and other tools built on sigrok.  In this repo, `wfmconvert sigrok ...` creates the session by exporting the waveform to an intermediate CSV representation and then handing that data to `sigrok-cli`.

If `sigrok-cli` is not installed in the current environment, the live export cells below print a short message instead of failing.

## Analog Session Example

This first example starts from a two-channel analog capture and creates a sigrok session archive.

In [ ]:
if not SIGROK_AVAILABLE:
    print("sigrok-cli is not installed in this environment, so the live .sr export is skipped.")
else:
    analog_dir = Path(tempfile.mkdtemp()) / "analog"
    analog_sr = create_sigrok_session("wfm/DS1102E-D.wfm", analog_dir)
    with zipfile.ZipFile(analog_sr) as zf:
        analog_members = sorted(zf.namelist())
    print(analog_sr)
    analog_members

In [ ]:
if not SIGROK_AVAILABLE:
    print("sigrok-cli is not installed in this environment, so there is no metadata to inspect.")
else:
    print(read_member_text(analog_sr, "version"))
    print(read_member_text(analog_sr, "metadata"))

## Logic Session Example

The same command can export a parsed digital trace.  Here we use the `DS1074Z-C.wfm` fixture, which exposes `D6`.

In [ ]:
if not SIGROK_AVAILABLE:
    print("sigrok-cli is not installed in this environment, so the live .sr export is skipped.")
else:
    logic_dir = Path(tempfile.mkdtemp()) / "logic"
    logic_sr = create_sigrok_session("wfm/DS1074Z-C.wfm", logic_dir)
    with zipfile.ZipFile(logic_sr) as zf:
        logic_members = sorted(zf.namelist())
    print(logic_sr)
    logic_members

In [ ]:
if not SIGROK_AVAILABLE:
    print("sigrok-cli is not installed in this environment, so there is no metadata to inspect.")
else:
    print(read_member_text(logic_sr, "version"))
    print(read_member_text(logic_sr, "metadata"))

## Command-Line Equivalent

The notebook helper above eventually runs the same CLI that you would use from a shell:

```bash
wfmconvert sigrok DS1102E-D.wfm
wfmconvert sigrok DS1074Z-C.wfm
```

The resulting `.sr` files can be opened directly in PulseView.